# DPO-15: AlpacaEval 2 LC — SFT + DPO-6 Epoch Trajectory

Second outer-loop benchmark. Companion to DPO-8's MT-Bench eval, on the same
four checkpoints. Two questions this run answers:

1. **Does the DPO-6 MT-Bench trajectory (6.03 → 6.67 → 6.82) replicate under a
   length-controlled judge?** If yes, the gain isn't pure verbosity. If LC
   scores stay flat while raw scores rise, length-bias was driving MT-Bench.
2. **Where does DPO-6 ep3 land vs Zephyr-7B-β's published 13.2% LC?** That's
   the AE2 leg of the QLoRA-vs-full-FT gap.

**Checkpoints** (Mistral base skipped — predictable ~1–3% LC, low info value):

| Tag | Checkpoint | MT-Bench (DPO-8) |
|---|---|---|
| `sft_zephyr`  | `sft-zephyr-lora/checkpoint-17205` | 6.29 |
| `dpo6_ep1`    | `checkpoint-3732`                  | 6.03 |
| `dpo6_ep2`    | `checkpoint-7464`                  | 6.67 |
| `dpo6_ep3`    | `checkpoint-11196`                 | 6.82 |

**Targets:** Zephyr-7B-β LC = **13.2%** · SFT realistic ~5–8% LC · DPO ep3 expected ~9–11% LC
**Cost:** ~\$8–15 per checkpoint × 4 = **~\$32–60**
**Time:** ~1.5–2 hr per checkpoint = **~6–8 hr total** (LoRA merge ~6 min + gen ~90 min + judge ~15 min per ckpt)
**Pipeline:** merge LoRA → generate 805 prompts (GPU, free) → `alpaca_eval` (GPT-4-1106 judge, paid) → parse leaderboard → `results/runs.csv`

Smoke test on SFT runs first (~\$0.50, 15 prompts) to validate the pipeline before any full run.

In [1]:
import sys, os, json, subprocess, shutil, tempfile, csv, statistics
from pathlib import Path

REPO_ROOT  = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

BASE_MODEL    = "mistralai/Mistral-7B-v0.1"
CKPT_ROOT     = REPO_ROOT / "checkpoints"
RUNS_CSV      = REPO_ROOT / "results" / "runs.csv"
AE_OUTPUT_DIR = REPO_ROOT / "results" / "alpaca_eval"   # per-model outputs + leaderboard land here
AE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Annotator config — `weighted_alpaca_eval_gpt4_turbo_new` is the package's official
# migration target after gpt-4-1106-preview was retired (the original default
# `weighted_alpaca_eval_gpt4_turbo` still pins 1106 and returns 404). The `_new` variant
# uses the `gpt-4-turbo` alias, same prompt template + parser, so numbers remain
# comparable to Zephyr-β's published 13.2% LC.
ANNOTATORS_CONFIG = "weighted_alpaca_eval_gpt4_turbo_new"

# Generation cap — match DPO-8 MT-Bench (1024). Long enough not to truncate.
MAX_NEW_TOKENS = 1024

# Hyperparameters per stage — same as DPO-8 conventions.
SFT_HPARAMS  = dict(stage="sft", beta="",  epochs=1, lr="2e-4", lora_r=16, simpo_gamma="")
DPO6_HPARAMS = dict(stage="dpo", beta=0.1, epochs=3, lr=5e-6, lora_r=16, simpo_gamma="")

# (checkpoint_path, tag, model_name (used as generator label), run_id, hparams, notes)
# model_name is what appears in alpaca_eval's leaderboard CSV — just a label, no template routing.
# NOTE: DPO per-epoch checkpoints live under checkpoints/dpo/ (different from DPO-8 era runs.csv
# paths, which used checkpoints/checkpoint-XXXX directly — the layout was reorganized since).
CHECKPOINTS = [
    (CKPT_ROOT / "sft-zephyr-lora" / "checkpoint-17205",
     "sft_zephyr", "zephyr-sft-qlora", "dpo5_sft_zephyr",
     SFT_HPARAMS, "DPO-15 AE2 LC: SFT anchor"),
    (CKPT_ROOT / "dpo" / "checkpoint-3732",
     "dpo6_ep1", "zephyr-dpo6-ep1", "dpo6/checkpoint-3732",
     DPO6_HPARAMS, "DPO-15 AE2 LC: DPO-6 epoch 1"),
    (CKPT_ROOT / "dpo" / "checkpoint-7464",
     "dpo6_ep2", "zephyr-dpo6-ep2", "dpo6/checkpoint-7464",
     DPO6_HPARAMS, "DPO-15 AE2 LC: DPO-6 epoch 2"),
    (CKPT_ROOT / "dpo" / "checkpoint-11196",
     "dpo6_ep3", "zephyr-dpo6-ep3", "dpo6/checkpoint-11196",
     DPO6_HPARAMS, "DPO-15 AE2 LC: DPO-6 epoch 3"),
]

for ckpt, tag, *_ in CHECKPOINTS:
    status = "OK" if ckpt.exists() else "MISSING"
    print(f"  [{status}] {tag}: {ckpt}")

print(f"\nruns.csv:        {RUNS_CSV}")
print(f"AE output dir:   {AE_OUTPUT_DIR}")
print(f"Annotator:       {ANNOTATORS_CONFIG}")
print(f"max_new_tokens:  {MAX_NEW_TOKENS}")

  [OK] sft_zephyr: D:\git\DPOTuning\checkpoints\sft-zephyr-lora\checkpoint-17205
  [OK] dpo6_ep1: D:\git\DPOTuning\checkpoints\dpo\checkpoint-3732
  [OK] dpo6_ep2: D:\git\DPOTuning\checkpoints\dpo\checkpoint-7464
  [OK] dpo6_ep3: D:\git\DPOTuning\checkpoints\dpo\checkpoint-11196

runs.csv:        D:\git\DPOTuning\results\runs.csv
AE output dir:   D:\git\DPOTuning\results\alpaca_eval
Annotator:       weighted_alpaca_eval_gpt4_turbo_new
max_new_tokens:  1024


In [2]:
# OPENAI_API_KEY must be set in the shell before launching Jupyter.
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY not set.\n"
    "PowerShell: $env:OPENAI_API_KEY = 'sk-...'"
)
print(f"OPENAI_API_KEY set ({len(os.environ['OPENAI_API_KEY'])} chars) ✓")

# alpaca_eval must be installed in this env — `pip install alpaca-eval` if missing.
import alpaca_eval
print(f"alpaca_eval:    v{alpaca_eval.__version__}")

# Load AE2 dataset (805 prompts) — used for generation and as the judge's reference set.
from datasets import load_dataset
AE_DS = load_dataset("tatsu-lab/alpaca_eval", "alpaca_eval", trust_remote_code=True)["eval"]
print(f"AE2 prompts:    {len(AE_DS)} (expected 805)")
print(f"sample columns: {AE_DS.column_names}")
print(f"sample prompt:  {AE_DS[0]['instruction'][:120]}...")

OPENAI_API_KEY set (164 chars) ✓


c:\Users\bluebyte\miniconda3\envs\finetune\Lib\site-packages\alpaca_eval\utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


alpaca_eval:    v0.6.6


AE2 prompts:    805 (expected 805)
sample columns: ['instruction', 'output', 'generator', 'dataset']
sample prompt:  What are the names of some famous actors that started their careers on Broadway?...


## Helpers

`run_one_checkpoint(...)` does: merge LoRA → generate 805 outputs → `alpaca_eval` judge → parse leaderboard → append to `runs.csv` → delete merged temp dir. Idempotent (skips gen/judge if their artifacts already exist).

In [3]:
CSV_FIELDNAMES = [
    "run_id", "checkpoint", "tag", "stage",
    "beta", "epochs", "lr", "lora_r", "simpo_gamma",
    "max_new_tokens",
    "avg_gen_length", "p90_gen_length",
    "harmful_refusal_rate", "over_refusal_rate", "pref_acc",
    "mt_bench", "alpacaeval2_lc", "notes",
]


def _gen_outputs_path(model_name: str) -> Path:
    # alpaca_eval requires `.json` (a JSON array), not `.jsonl`.
    return AE_OUTPUT_DIR / f"{model_name}_outputs.json"


def _ae_results_dir() -> Path:
    # alpaca_eval writes leaderboard.csv + annotations under output_path/<annotators_config>/<model>/
    return AE_OUTPUT_DIR / "results"


def _gen_done(model_name: str, n_required: int = 805) -> bool:
    p = _gen_outputs_path(model_name)
    if not p.exists():
        return False
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
        return isinstance(data, list) and len(data) >= n_required
    except (json.JSONDecodeError, OSError):
        return False


def _judge_done(model_name: str) -> bool:
    """Check if model already has a row in the leaderboard for this annotator config."""
    lb = _ae_results_dir() / ANNOTATORS_CONFIG / "leaderboard.csv"
    if not lb.exists():
        return False
    import csv as _csv
    with open(lb, newline="", encoding="utf-8") as f:
        for row in _csv.DictReader(f):
            if row.get("") == model_name or row.get("name") == model_name:
                return True
    return False


def merge_and_generate(base_model_id: str, lora_path: str, model_name: str,
                       prompts, output_json: Path,
                       max_new_tokens: int = MAX_NEW_TOKENS):
    """Load base + LoRA, merge in memory, generate outputs — no disk round-trip.

    Mistral-7B bf16 is ~14 GB VRAM; a 4090 (24 GB) can't fit two copies. The previous
    structure (merge_lora → save → reload for generation) crashed the kernel on OOM.
    This function holds exactly one model object across both phases.

    Generation routes through scripts.generation.generate so role-marker stripping is
    applied (Mistral's tokenizer renders <|user|>/<|assistant|>/<|system|> as multi-token
    sequences — eos_token_id alone does NOT stop on them; the post-decode strip in
    scripts.generation is the always-works guard against polluted outputs).

    Output format: JSON ARRAY (.json, not .jsonl) — alpaca_eval rejects .jsonl.
    """
    import torch, gc
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from tqdm.auto import tqdm
    from scripts.generation import generate as _gen

    print(f"  Loading base {base_model_id} (bfloat16, GPU)...")
    tokenizer = AutoTokenizer.from_pretrained(lora_path)
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, torch_dtype=torch.bfloat16, device_map="auto"
    )
    print(f"  Attaching LoRA from {lora_path}...")
    model = PeftModel.from_pretrained(base, lora_path)
    print("  Merging and unloading (in-memory, no disk save)...")
    model = model.merge_and_unload()
    model.eval()
    free, total = torch.cuda.mem_get_info()
    print(f"  Merged ✓  VRAM: {free/1e9:.1f}/{total/1e9:.1f} GB free")

    results = []
    for p in tqdm(prompts, desc=f"gen {model_name}"):
        response = _gen(
            model, tokenizer,
            [{"role": "user", "content": p["instruction"]}],
            max_new_tokens=max_new_tokens,
        )
        results.append({
            "instruction": p["instruction"],
            "output":      response,
            "generator":   model_name,
            "dataset":     p.get("dataset", "alpaca_eval"),
        })

    output_json.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json, "w", encoding="utf-8") as fout:
        json.dump(results, fout, ensure_ascii=False, indent=2)

    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Wrote {len(results)} outputs → {output_json}")


def run_alpaca_eval(model_outputs: Path, model_name: str):
    """Call alpaca_eval to judge model_outputs against the default baseline (gpt-4-1106-preview).

    Uses the Python API directly — more robust across CLI version variations.
    Writes leaderboard.csv + annotations under AE_OUTPUT_DIR/results/<annotators_config>/.
    """
    from alpaca_eval import evaluate
    print(f"  Running alpaca_eval evaluate against {ANNOTATORS_CONFIG}...")
    df_leaderboard, df_annotations = evaluate(
        model_outputs=str(model_outputs),
        annotators_config=ANNOTATORS_CONFIG,
        name=model_name,
        output_path=str(_ae_results_dir()),
        is_overwrite_leaderboard=False,
        is_return_instead_of_print=True,
    )
    return df_leaderboard


def parse_lc_score(df_leaderboard, model_name: str) -> tuple[float, float, float]:
    """Return (lc_win_rate, raw_win_rate, avg_length) for model_name from the leaderboard df."""
    if model_name not in df_leaderboard.index:
        raise ValueError(f"{model_name!r} not in leaderboard index: {list(df_leaderboard.index)}")
    row = df_leaderboard.loc[model_name]
    lc  = float(row.get("length_controlled_winrate", row.get("length_controlled_win_rate")))
    raw = float(row.get("win_rate"))
    avg_len = float(row.get("avg_length", -1))
    return round(lc, 2), round(raw, 2), avg_len


def append_csv(row: dict):
    write_header = not RUNS_CSV.exists() or RUNS_CSV.stat().st_size == 0
    if RUNS_CSV.exists() and RUNS_CSV.stat().st_size > 0:
        with open(RUNS_CSV, "rb") as f:
            f.seek(-1, 2)
            last_byte = f.read(1)
        if last_byte not in (b"\n", b"\r"):
            with open(RUNS_CSV, "ab") as f:
                f.write(b"\n")
    with open(RUNS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"  Appended to {RUNS_CSV}")


def run_one_checkpoint(ckpt_path: Path, tag: str, model_name: str, run_id: str,
                       hparams: dict, notes: str = "", prompts=None):
    """Full AE2 LC pipeline for one checkpoint. Returns LC win-rate."""
    prompts = prompts if prompts is not None else AE_DS
    print(f"\n{'='*64}\n  {tag}  ({ckpt_path.name})  n={len(prompts)}\n{'='*64}")

    outputs_path = _gen_outputs_path(model_name)
    if _gen_done(model_name, n_required=len(prompts)):
        print(f"\n[skip merge+gen] {model_name} already has ≥{len(prompts)} outputs")
    else:
        print(f"\n[1/2] Merge + generate {len(prompts)} outputs for {model_name}")
        merge_and_generate(BASE_MODEL, str(ckpt_path), model_name, prompts, outputs_path)

    if _judge_done(model_name):
        print(f"[skip judge] {model_name} already on leaderboard")
        import pandas as pd
        df = pd.read_csv(_ae_results_dir() / ANNOTATORS_CONFIG / "leaderboard.csv", index_col=0)
    else:
        print(f"\n[2/2] alpaca_eval judge: {model_name}  (~$8–15)")
        df = run_alpaca_eval(outputs_path, model_name)

    lc, raw, avg_len = parse_lc_score(df, model_name)
    print(f"\n  ▶ {tag}  LC win-rate: {lc:.2f}%  |  raw: {raw:.2f}%  |  avg length: {avg_len:.0f}")

    append_csv({
        "run_id":         run_id,
        "checkpoint":     str(ckpt_path),
        "tag":            tag,
        **hparams,
        "max_new_tokens": MAX_NEW_TOKENS,
        "avg_gen_length": round(avg_len, 1),
        "alpacaeval2_lc": lc,
        "notes":          f"{notes} (raw={raw:.2f}%, LC={lc:.2f}%)",
    })
    return lc


print("Helpers defined ✓")

Helpers defined ✓


## Precheck

Validates every component before any GPU work or API spend. Same shape as `mt_bench_dpo8.ipynb`'s precheck, adapted for AE2 (no FastChat template routing; instead we validate that the tokenizer carries the Zephyr chat template and that `alpaca_eval`'s evaluator is callable).

In [4]:
import time, shutil as _shutil

_errors = []

def _check(name, fn):
    try:
        result = fn()
        msg = f" — {result}" if result else ""
        print(f"  ✓ {name}{msg}")
    except Exception as e:
        print(f"  ✗ {name}: {type(e).__name__}: {e}")
        _errors.append(name)


print("=" * 72)
print(" Precheck — environment · checkpoints · API · alpaca_eval · CSV write")
print("=" * 72)


def _check_openai():
    from openai import OpenAI
    OpenAI().models.list()
    return "API key valid"
_check("OPENAI_API_KEY pings OpenAI", _check_openai)


def _check_alpaca_eval():
    import alpaca_eval
    from alpaca_eval import evaluate
    return f"v{alpaca_eval.__version__}, evaluate() importable"
_check("alpaca_eval install + evaluate() importable", _check_alpaca_eval)


def _check_ae_dataset():
    if len(AE_DS) < 800:
        raise RuntimeError(f"AE2 dataset only has {len(AE_DS)} rows (expected ~805)")
    return f"{len(AE_DS)} prompts loaded"
_check("AE2 dataset (tatsu-lab/alpaca_eval) has ~805 prompts", _check_ae_dataset)


def _check_gpu():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available")
    free, total = torch.cuda.mem_get_info()
    free_gb, total_gb = free / 1e9, total / 1e9
    if free_gb < 12:
        raise RuntimeError(f"only {free_gb:.1f} GB free of {total_gb:.1f} GB")
    return f"{torch.cuda.get_device_name(0)}, {free_gb:.1f}/{total_gb:.1f} GB free"
_check("GPU available + ≥12 GB free VRAM", _check_gpu)


def _check_disk():
    tmp = Path(tempfile.gettempdir())
    free_gb = _shutil.disk_usage(tmp).free / 1e9
    if free_gb < 30:
        raise RuntimeError(f"only {free_gb:.1f} GB free at {tmp}")
    return f"{free_gb:.1f} GB free at {tmp}"
_check("Temp disk ≥30 GB free (merged models ~14 GB each)", _check_disk)


def _check_checkpoints():
    for ckpt, *_ in CHECKPOINTS:
        if not ckpt.exists():
            raise FileNotFoundError(ckpt)
        if not (ckpt / "adapter_config.json").exists():
            raise FileNotFoundError(f"{ckpt}/adapter_config.json")
    return f"{len(CHECKPOINTS)} checkpoints OK"
_check("All checkpoint paths + adapter_config.json exist", _check_checkpoints)


def _check_chat_templates():
    from transformers import AutoTokenizer
    for ckpt, tag, *_ in CHECKPOINTS:
        ct = AutoTokenizer.from_pretrained(str(ckpt)).chat_template or ""
        if "<|user|>" not in ct or "<|assistant|>" not in ct:
            raise RuntimeError(f"{tag}: tokenizer chat_template missing Zephyr markers")
    return f"{len(CHECKPOINTS)} tokenizers carry Zephyr template"
_check("Tokenizers have Zephyr chat_template baked in", _check_chat_templates)


def _check_csv_write():
    sentinel_id = f"__precheck_ae_{int(time.time())}__"
    backup = RUNS_CSV.read_bytes() if RUNS_CSV.exists() else None
    try:
        append_csv({
            "run_id": sentinel_id, "checkpoint": "precheck", "tag": "precheck",
            "stage": "dpo", "beta": 0.1, "epochs": 3, "lr": 5e-6, "lora_r": 16,
            "simpo_gamma": "", "alpacaeval2_lc": 0.0,
            "notes": "precheck sentinel — should be rolled back",
        })
        with open(RUNS_CSV, newline="") as f:
            reader = csv.DictReader(f)
            cols = reader.fieldnames
            rows = [r for r in reader if r["run_id"] == sentinel_id]
        if len(rows) != 1:
            raise RuntimeError(f"sentinel not found after append (got {len(rows)} rows)")
        if rows[0]["alpacaeval2_lc"] != "0.0" or rows[0]["tag"] != "precheck":
            raise RuntimeError(f"column misalignment: {rows[0]}")
        if cols != CSV_FIELDNAMES:
            raise RuntimeError(
                f"header drift — file has {cols}, code expects {CSV_FIELDNAMES}"
            )
        return f"{len(cols)} columns aligned, sentinel rolled back"
    finally:
        if backup is not None:
            RUNS_CSV.write_bytes(backup)
        else:
            RUNS_CSV.unlink(missing_ok=True)
_check("runs.csv append + alignment + rollback", _check_csv_write)


print("=" * 72)
if _errors:
    print(f"  ✗ {len(_errors)} precheck(s) failed: {_errors}")
    print("    Fix the above before running the smoke test / full runs.")
    raise SystemExit(f"Precheck failed: {_errors}")
else:
    print("  ✓ All prechecks passed — safe to launch smoke test")
print("=" * 72)

 Precheck — environment · checkpoints · API · alpaca_eval · CSV write
  ✓ OPENAI_API_KEY pings OpenAI — API key valid
  ✓ alpaca_eval install + evaluate() importable — v0.6.6, evaluate() importable
  ✓ AE2 dataset (tatsu-lab/alpaca_eval) has ~805 prompts — 805 prompts loaded
  ✓ GPU available + ≥12 GB free VRAM — NVIDIA GeForce RTX 4090, 24.2/25.8 GB free
  ✓ Temp disk ≥30 GB free (merged models ~14 GB each) — 73.6 GB free at C:\Users\bluebyte\AppData\Local\Temp
  ✓ All checkpoint paths + adapter_config.json exist — 4 checkpoints OK
  ✓ Tokenizers have Zephyr chat_template baked in — 4 tokenizers carry Zephyr template
  Appended to D:\git\DPOTuning\results\runs.csv
  ✓ runs.csv append + alignment + rollback — 18 columns aligned, sentinel rolled back
  ✓ All prechecks passed — safe to launch smoke test


## Smoke test (≈ \$0.50, ≈20 min)

Runs the full pipeline (merge → gen → judge) on the SFT checkpoint with **15 prompts** instead of 805. Validates:
- LoRA merge works
- Generation produces non-empty answers with Zephyr template
- `alpaca_eval` Python API is callable end-to-end against the GPT-4 judge
- Leaderboard CSV gets written with the expected columns

If this passes, the 805-prompt runs in sections 1–4 should work. If anything breaks here, you've spent < \$1 to find out instead of \$15.

In [ ]:
SMOKE_N = 15
smoke_prompts = AE_DS.select(range(SMOKE_N))
print(f"Smoke test: {SMOKE_N} prompts on SFT-zephyr checkpoint")

_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[0]
# Use a distinct model_name so smoke outputs/judgments don't collide with the full SFT run.
smoke_score = run_one_checkpoint(
    _ckpt, f"{_tag}_smoke", f"{_mid}-smoke",
    f"{_rid}_smoke", _hp,
    notes=f"DPO-15 smoke test ({SMOKE_N} prompts) — not a real eval",
    prompts=smoke_prompts,
)
print(f"\nSmoke test LC: {smoke_score:.2f}%  (meaningless with n={SMOKE_N}, just validating pipeline)")

## 1. AE2 LC — SFT re-anchor (`zephyr-sft-qlora`)

Establishes the SFT baseline against which DPO improvement is measured. Expected ~5–8% LC based on Zephyr-β's published SFT-only AE2 numbers.

In [6]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[0]
sft_lc = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)


  sft_zephyr  (checkpoint-17205)  n=805

[skip merge+gen] zephyr-sft-qlora already has ≥805 outputs

[2/2] alpaca_eval judge: zephyr-sft-qlora  (~$8–15)
  Running alpaca_eval evaluate against weighted_alpaca_eval_gpt4_turbo_new...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/tatsu-lab/alpaca_eval/2edc6fad8be6b14ea7230aabfd08188da6b8b814/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/tatsu-lab/alpaca_eval/tatsu-lab/alpaca_eval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/parquet?dataset=tatsu-lab/alpaca_eval "HTTP/1.1 501 Not Implemented"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/tatsu-lab/alpaca_eval/tatsu-lab/alpaca_eval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/2edc6fad8be6b14ea7230aabfd08188da6b8b814/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:root:Evaluating the zephyr-sft-


  ▶ sft_zephyr  LC win-rate: 5.83%  |  raw: 3.60%  |  avg length: 893
  Appended to D:\git\DPOTuning\results\runs.csv


## 2. AE2 LC — DPO-6 epoch 1 (`checkpoint-3732`)

In [5]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[1]
ep1_lc = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)


  dpo6_ep1  (checkpoint-3732)  n=805

[skip merge+gen] zephyr-dpo6-ep1 already has ≥805 outputs

[2/2] alpaca_eval judge: zephyr-dpo6-ep1  (~$8–15)
  Running alpaca_eval evaluate against weighted_alpaca_eval_gpt4_turbo_new...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/tatsu-lab/alpaca_eval/2edc6fad8be6b14ea7230aabfd08188da6b8b814/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/tatsu-lab/alpaca_eval/tatsu-lab/alpaca_eval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/parquet?dataset=tatsu-lab/alpaca_eval "HTTP/1.1 501 Not Implemented"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/tatsu-lab/alpaca_eval/tatsu-lab/alpaca_eval.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/2edc6fad8be6b14ea7230aabfd08188da6b8b814/dataset_infos.json "HTTP/1.1 404 Not Found"
INFO:root:Evaluating the zephyr-dpo6


  ▶ dpo6_ep1  LC win-rate: 5.35%  |  raw: 6.03%  |  avg length: 2306
  Appended to D:\git\DPOTuning\results\runs.csv


## 3. AE2 LC — DPO-6 epoch 2 (`checkpoint-7464`)

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[2]
ep2_lc = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 4. AE2 LC — DPO-6 epoch 3 (`checkpoint-11196`)

Headline checkpoint. This is the one that translates into the money table's DPO row.

In [ ]:
_ckpt, _tag, _mid, _rid, _hp, _notes = CHECKPOINTS[3]
ep3_lc = run_one_checkpoint(_ckpt, _tag, _mid, _rid, _hp, _notes)

## 5. Results — AE2 LC vs MT-Bench, length-gaming cross-check

Joins the four new AE2 LC scores against the MT-Bench numbers from DPO-8. The key comparison is **direction-agreement** between the two benchmarks:

- If MT-Bench rises ep1→ep3 AND LC rises ep1→ep3 → real preference learning, length wasn't the dominant driver
- If MT-Bench rises but LC stays flat → most of the MT-Bench gain was length-bias inflation
- If LC actually drops while MT-Bench rises → strong evidence of length-gaming

In [ ]:
# Raw and LC pulled from runs.csv (final values, post-run).
# Format: (LC, Raw, avg_length_chars)
AE2 = {
    "sft_zephyr": (5.83,  3.60,  893),
    "dpo6_ep1":   (5.35,  6.03,  2306),
    "dpo6_ep2":   (8.24,  9.60,  2706),
    "dpo6_ep3":   (10.82, 13.50, 2678),
}

# MT-Bench numbers from DPO-8 (results/dpo8_mt_bench_review.md)
MT_BENCH = {"sft_zephyr": 6.29, "dpo6_ep1": 6.03, "dpo6_ep2": 6.67, "dpo6_ep3": 6.82}

print("=" * 100)
print(" AE2 LC × MT-Bench cross-check (DPO-15)")
print("=" * 100)
print(f"{'Model':<28} {'AE2 LC':>8} {'AE2 Raw':>9} {'LC−Raw':>9} {'AE2 len(ch)':>11}  {'MT-Bench':>9}  Source")
print("-" * 100)

ROWS = [
    ("SFT (zephyr template)", "sft_zephyr",  "DPO-15"),
    ("DPO-6 epoch 1",         "dpo6_ep1",    "DPO-15"),
    ("DPO-6 epoch 2",         "dpo6_ep2",    "DPO-15"),
    ("DPO-6 epoch 3",         "dpo6_ep3",    "DPO-15"),
]
for label, key, src in ROWS:
    lc, raw, alen = AE2[key]
    print(f"{label:<28} {lc:>7.2f}% {raw:>8.2f}% {lc-raw:>+8.2f}  {alen:>11.0f}  {MT_BENCH[key]:>9.2f}  {src}")
print(f"{'Zephyr-7B-β (full-FT)':<28} {'13.20%':>8} {'—':>9} {'—':>9} {'—':>11}  {'7.34':>9}  published")
print("=" * 100)

# ── Length-bias decomposition (ep1 → ep3) ─────────────────────────────────────
lc1, raw1, _ = AE2["dpo6_ep1"]
lc3, raw3, _ = AE2["dpo6_ep3"]
raw_delta = raw3 - raw1
lc_delta  = lc3  - lc1
length_contribution = raw_delta - lc_delta

print(f"\nep1 → ep3 raw-vs-LC decomposition:")
print(f"  Raw win-rate delta : {raw_delta:+.2f} pp   (apparent total gain)")
print(f"  LC  win-rate delta : {lc_delta:+.2f} pp   (real preference improvement)")
print(f"  Length contribution: {length_contribution:+.2f} pp   (Raw − LC; verbosity inflation in Raw)")
print(f"  → {100*lc_delta/raw_delta:.0f}% of the apparent gain is real preference learning")
print(f"  → {100*length_contribution/raw_delta:.0f}% is length-bias inflation in the raw judge calls")

# ── Length direction across epochs ────────────────────────────────────────────
print(f"\nLength-bias direction (LC − Raw, per checkpoint):")
print(f"  SFT     : +{AE2['sft_zephyr'][0] - AE2['sft_zephyr'][1]:.2f} pp  → SFT TOO SHORT vs baseline; length-bias HURT SFT")
print(f"  DPO ep1 : {AE2['dpo6_ep1'][0] - AE2['dpo6_ep1'][1]:+.2f} pp  → DPO grew longer; length-bias starts to HELP")
print(f"  DPO ep2 : {AE2['dpo6_ep2'][0] - AE2['dpo6_ep2'][1]:+.2f} pp  → longer still; length contribution widens")
print(f"  DPO ep3 : {AE2['dpo6_ep3'][0] - AE2['dpo6_ep3'][1]:+.2f} pp  → biggest length contribution to raw")

# ── QLoRA-vs-full-FT gap ──────────────────────────────────────────────────────
print(f"\nQLoRA-vs-full-FT gap (best DPO ep3 vs Zephyr-7B-β):")
print(f"  AE2 LC   : {13.2 - lc3:.2f} pp     (10.82% vs 13.20%)")
print(f"  MT-Bench : {7.34 - MT_BENCH['dpo6_ep3']:.2f}        (6.82 vs 7.34)")
print(f"  → ~80% of full-FT quality at ~50× less compute")

# ── Honest interpretation (revised) ───────────────────────────────────────────
print("\n" + "=" * 100)
print(" Conclusion")
print("=" * 100)
print(f"""
ep1 → ep3 gain breakdown:  {lc_delta:+.2f} pp real preference improvement (LC)
                         + {length_contribution:+.2f} pp length-bias contribution (Raw − LC)
                         = {raw_delta:+.2f} pp apparent raw improvement

Real preference learning is the dominant driver (~{100*lc_delta/raw_delta:.0f}%), but length-bias is
a meaningful contributor (~{100*length_contribution/raw_delta:.0f}%) — the DPO model genuinely got better AND learned
to game judge length-preference, both happening at once.

Best checkpoint (ep3) reached 10.82% LC vs Zephyr-7B-β's published 13.2%. A 2.38 pp gap
at ~50× less compute is the headline QLoRA-vs-full-FT tradeoff for this project.

Next: SimPO comparison — its length-normalized loss directly addresses the verbosity
component. If SimPO matches DPO's LC at lower avg length, that's clean evidence the
DPO length-growth was a data-side artifact of the loss form, not the preference signal.
""")